# 04 - Smooth Approximations and Piecewise Linear Functions

This notebook explores smooth approximations of piecewise linear functions and their role in understanding loss landscapes.

**Converted from:** `sgd_example_2.nb` and `Danilo_piecewiselinlosses.nb` (Mathematica)

## Contents:
1. Piecewise linear loss functions
2. Smooth sigmoid approximations
3. Comparison of optimization dynamics
4. Effect of smoothness on convergence

## Background

Many modern loss functions (e.g., ReLU networks) are piecewise linear. Understanding how smooth approximations relate to non-smooth functions helps us:
- Analyze optimization landscapes theoretically
- Understand the role of activation functions
- Bridge discrete and continuous perspectives

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd() / 'utils'))

from sgd_simulator import SGDSimulator, SGDConfig
from loss_functions import (
    piecewise_linear_loss,
    piecewise_linear_gradient,
    smooth_approximation,
    generate_noisy_data,
    PiecewiseLinearLoss,
    SmoothNonlinearLoss
)
from visualization import (
    plot_loss_landscape,
    plot_parameter_evolution,
    plot_trajectories
)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✓ Libraries imported successfully!")

## 1. Piecewise Linear Functions

Let's start by examining simple piecewise linear functions and their properties.

In [ ]:
# Generate data for linear regression
np.random.seed(42)

# Simple linear relationship with noise
n_points = 50
x_data = np.linspace(-2, 2, n_points)
true_intercept = 1.0
true_slope = 0.5
noise_std = 0.3

y_data = true_intercept + true_slope * x_data + np.random.normal(0, noise_std, n_points)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x_data, y_data, alpha=0.6, s=50, label='Data')
ax.plot(x_data, true_intercept + true_slope * x_data, 'r-', 
       linewidth=2, label=f'True: y = {true_intercept} + {true_slope}x')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Linear Regression Data', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"True parameters: intercept={true_intercept}, slope={true_slope}")

In [ ]:
# Visualize piecewise linear loss landscape
loss_piecewise = PiecewiseLinearLoss()

# Parameter range
param_range = ((-0.5, 2.5), (-0.5, 1.5))

fig, ax = plt.subplots(figsize=(12, 10))
plot_loss_landscape(
    loss_fn=loss_piecewise,
    x_data=x_data,
    y_data=y_data,
    param_range=param_range,
    n_points=100,
    contour_levels=40,
    ax=ax,
    title='Piecewise Linear Loss Landscape (MSE)'
)

# Mark true minimum
ax.plot(true_slope, true_intercept, 'r*', markersize=20,
       markeredgecolor='black', markeredgewidth=2, label='True minimum')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("Piecewise linear (quadratic MSE) loss has a single global minimum")

## 2. Smooth Approximations

Now let's compare smooth (sigmoid-based) approximations with different steepness parameters.

The smooth approximation is:
$$f(x; a, b) = x \left(1 + \frac{b}{1 + e^{-ax}}\right)$$

As $a \to \infty$, this approaches a piecewise linear function.

In [ ]:
# Compare different smoothness levels
x_plot = np.linspace(-3, 3, 200)

# Different steepness parameters
a_values = [0.5, 1.0, 2.0, 5.0, 10.0]
b_value = 1.0

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, a in enumerate(a_values):
    ax = axes[idx]
    
    # Compute smooth approximation
    y_smooth = smooth_approximation(x_plot, a=a, b=b_value)
    
    # Plot
    ax.plot(x_plot, y_smooth, 'b-', linewidth=2.5, label=f'a={a}')
    ax.plot(x_plot, x_plot, 'r--', linewidth=2, alpha=0.5, label='Linear (y=x)')
    ax.plot(x_plot, x_plot * (1 + b_value), 'g--', linewidth=2, 
           alpha=0.5, label=f'Linear (y=x(1+{b_value}))')
    
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('f(x)', fontsize=11)
    ax.set_title(f'Smooth Approximation (a={a}, b={b_value})', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Remove extra subplot
axes[-1].remove()

plt.tight_layout()
plt.show()

print("As 'a' increases, the function transitions more sharply")
print("This represents a smooth approximation to piecewise behavior")

## 3. SGD on Smooth vs Piecewise Functions

Let's run SGD on both smooth and piecewise linear loss functions and compare the dynamics.

In [ ]:
# Generate data with nonlinear pattern
np.random.seed(123)

x_data_nl, y_data_nl = generate_noisy_data(
    x_range=(-3, 3),
    n_points=15,
    n_samples_per_point=3,
    noise_std=0.4,
    p=1.0,
    random_state=123
)

# Create loss functions
loss_smooth = SmoothNonlinearLoss(p=1.0)
loss_piecewise = PiecewiseLinearLoss()

print(f"Data: {len(x_data_nl)} points for nonlinear fitting")

In [ ]:
# Run SGD on smooth loss
sgd_config = SGDConfig(
    learning_rate=0.04,
    batch_size=10,
    n_iterations=2000,
    random_state=42
)

initial_params = np.array([0.5, 0.5])

# Smooth loss optimization
simulator_smooth = SGDSimulator(sgd_config)
traj_smooth, iters_smooth = simulator_smooth.run_trajectory(
    initial_params=initial_params,
    gradient_fn=loss_smooth.gradient,
    x_data=x_data_nl,
    y_data=y_data_nl,
    save_every=10
)

print("✓ SGD on smooth loss completed")
print(f"  Final params: {traj_smooth[-1]}")
print(f"  Final loss: {loss_smooth(traj_smooth[-1], x_data_nl, y_data_nl):.4f}")

In [ ]:
# Visualize smooth loss optimization
param_range = ((-1, 3), (-1, 3))

fig, ax = plt.subplots(figsize=(12, 10))
plot_loss_landscape(
    loss_fn=loss_smooth,
    x_data=x_data_nl,
    y_data=y_data_nl,
    param_range=param_range,
    n_points=80,
    trajectory=traj_smooth,
    contour_levels=35,
    ax=ax,
    title='SGD on Smooth Nonlinear Loss'
)
plt.tight_layout()
plt.show()

In [ ]:
# Compare parameter evolution
fig, ax = plt.subplots(figsize=(14, 7))

plot_parameter_evolution(
    iterations=iters_smooth,
    trajectory=traj_smooth,
    param_names=['Parameter a (smooth)', 'Parameter b (smooth)'],
    ax=ax,
    title='Parameter Evolution on Smooth Loss'
)

plt.tight_layout()
plt.show()

## 4. Effect of Smoothness on Convergence

Let's study how the smoothness parameter affects convergence speed and final solution.

In [ ]:
# Test different smoothness levels
# We'll vary the implicit smoothness by changing learning rate and batch size

learning_rates = [0.01, 0.03, 0.05, 0.08, 0.10]
batch_sizes = [5, 10, 20]

results = {}

print("Testing different configurations...")
for bs in batch_sizes:
    results[bs] = []
    
    for lr in learning_rates:
        config = SGDConfig(
            learning_rate=lr,
            batch_size=bs,
            n_iterations=1500,
            random_state=42
        )
        
        sim = SGDSimulator(config)
        traj, iters = sim.run_trajectory(
            initial_params=initial_params,
            gradient_fn=loss_smooth.gradient,
            x_data=x_data_nl,
            y_data=y_data_nl,
            save_every=10
        )
        
        final_loss = loss_smooth(traj[-1], x_data_nl, y_data_nl)
        results[bs].append(final_loss)

print("✓ Completed parameter sweep")

In [ ]:
# Plot results
fig, ax = plt.subplots(figsize=(12, 7))

for bs in batch_sizes:
    ax.plot(learning_rates, results[bs], 'o-', linewidth=2, 
           markersize=8, label=f'Batch size = {bs}', alpha=0.8)

ax.set_xlabel('Learning Rate', fontsize=12)
ax.set_ylabel('Final Loss', fontsize=12)
ax.set_title('Final Loss vs Learning Rate for Different Batch Sizes', fontsize=14)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Optimal learning rate depends on batch size!")

## 5. Comparative Visualization

Let's create a side-by-side comparison of different approximation approaches.

In [ ]:
# Multiple trajectories with different initializations
np.random.seed(456)
n_traj = 4

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Different starting points
init_points = [
    np.array([0.3, 0.3]),
    np.array([2.0, 0.5]),
    np.array([1.0, 2.0]),
    np.array([0.5, 1.5])
]

param_range_tight = ((-0.5, 2.5), (-0.5, 2.5))

for idx, (ax, init_p) in enumerate(zip(axes.flatten(), init_points)):
    # Run SGD
    config = SGDConfig(learning_rate=0.05, batch_size=8, 
                      n_iterations=1500, random_state=42+idx)
    sim = SGDSimulator(config)
    traj, _ = sim.run_trajectory(
        init_p, loss_smooth.gradient, x_data_nl, y_data_nl, save_every=8
    )
    
    # Plot
    plot_loss_landscape(
        loss_fn=loss_smooth,
        x_data=x_data_nl,
        y_data=y_data_nl,
        param_range=param_range_tight,
        n_points=70,
        trajectory=traj,
        contour_levels=25,
        ax=ax,
        title=f'Init: [{init_p[0]:.1f}, {init_p[1]:.1f}]'
    )

plt.suptitle('SGD from Different Initializations', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

## 6. Loss Function Shape Analysis

Let's analyze the curvature and gradient properties of smooth vs piecewise linear functions.

In [ ]:
# Analyze gradient magnitude along a line
t_values = np.linspace(-2, 2, 100)
line_start = np.array([0.0, 0.0])
line_end = np.array([2.0, 2.0])

gradients_smooth = []
losses_smooth = []

for t in t_values:
    params = line_start + t * (line_end - line_start)
    grad = loss_smooth.gradient(params, x_data_nl, y_data_nl)
    loss_val = loss_smooth(params, x_data_nl, y_data_nl)
    
    gradients_smooth.append(np.linalg.norm(grad))
    losses_smooth.append(loss_val)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Loss along line
axes[0].plot(t_values, losses_smooth, 'b-', linewidth=2.5)
axes[0].set_xlabel('t (parameter along line)', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Loss Along Line in Parameter Space', fontsize=13)
axes[0].grid(True, alpha=0.3)

# Gradient magnitude
axes[1].plot(t_values, gradients_smooth, 'r-', linewidth=2.5)
axes[1].set_xlabel('t (parameter along line)', fontsize=12)
axes[1].set_ylabel('Gradient Magnitude', fontsize=12)
axes[1].set_title('Gradient Magnitude Along Line', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Smooth losses have smooth gradients (no discontinuities)")

## Summary

In this notebook, we explored:

1. **Piecewise Linear Functions**: Simple MSE loss with linear predictions
2. **Smooth Approximations**: Sigmoid-based smooth transitions
3. **Optimization Comparison**: SGD behavior on smooth vs non-smooth functions
4. **Smoothness Effects**: How approximation quality affects convergence
5. **Gradient Analysis**: Smooth functions have continuous gradients

**Key Insights:**
- Smooth approximations make theoretical analysis tractable
- The smoothness parameter controls the transition sharpness
- SGD converges smoothly on smooth losses (no gradient discontinuities)
- Learning rate and batch size interact with loss landscape geometry

**Practical Relevance:**
- Modern neural networks use smooth activations (sigmoid, tanh, swish, GELU)
- ReLU is piecewise linear but smooth almost everywhere
- Understanding smooth approximations helps design better architectures

**Next Steps:**
- Notebook 5 will focus specifically on 2D SDE escape time analysis
- We'll dive deeper into the mathematical theory of escape times